# 🏦 Credit Card Fraud Detection — ML Analysis
### Basics of AI and Machine Learning — Final Project
**Dataset:** Credit Card Transactions (Western United States)  
**Task:** Binary Classification — Predict whether a transaction is fraudulent (`is_fraud`)  
**Business Goal:** The model should err on the side of caution — it is better to flag a legitimate transaction as fraud than to miss actual fraud. This means we prioritise **Recall** (catching real fraud) over Precision.

---
**Notebook Structure:**
1. Data Loading & Initial Inspection
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing & Feature Engineering
4. Train–Test Split
5. Model Training & Hyperparameter Tuning
6. Evaluation & Comparison
7. Model Interpretation & Feature Importance
8. Conclusions


In [ ]:
# ─── Import all required libraries ───────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import kagglehub

# Sklearn — preprocessing & splitting
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Sklearn — models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# XGBoost & LightGBM (bonus models)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Sklearn — metrics
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc,
    accuracy_score, f1_score, precision_score, recall_score
)

# Imbalanced-learn — handle class imbalance
from imblearn.over_sampling import SMOTE

# Plotting settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (10, 5)})

print("✅ All libraries imported successfully.")


Path to dataset files: /Users/ziemyks/.cache/kagglehub/datasets/dhruvb2028/credit-card-fraud-dataset/versions/1
Shape: (339607, 15)


,trans_date_trans_time,merchant,category,amt,city,state,lat,long,city_pop,job,dob,trans_num,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:44,"Heller, Gutmann and Zieme",grocery_pos,107.23,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,49.159047,-118.186462,0
1,2019-01-01 00:00:51,Lind-Buckridge,entertainment,220.11,Malad City,ID,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,43.150704,-112.154481,0
2,2019-01-01 00:07:27,Kiehn Inc,grocery_pos,96.29,Grenada,CA,41.6125,-122.5258,589,Systems analyst,1945-12-21,413636e759663f264aae1819a4d4f231,41.657520,-122.230347,0
3,2019-01-01 00:09:03,Beier-Hyatt,shopping_pos,7.77,High Rolls Mountain Park,NM,32.9396,-105.8189,899,Naval architect,1967-08-30,8a6293af5ed278dea14448ded2685fea,32.863258,-106.520205,0
4,2019-01-01 00:21:32,Bruen-Yost,misc_pos,6.85,Freedom,WY,43.0172,-111.0292,471,"Education officer, museum",1967-08-02,f3c43d336e92a44fc2fb67058d5949e3,43.753735,-111.454923,0


## 1. 📥 Data Loading & Initial Inspection

We load the dataset downloaded via `kagglehub`. The dataset contains credit card transactions from the western United States with 15 columns including customer details, merchant info, transaction amount, and the fraud label.


In [ ]:
# ─── Load dataset ─────────────────────────────────────────────────────────────
path = kagglehub.dataset_download("dhruvb2028/credit-card-fraud-dataset")
csv_file = os.path.join(path, "credit_card_frauds.csv")
df = pd.read_csv(csv_file)

print(f"Dataset path : {csv_file}")
print(f"Shape        : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn names :\n{df.columns.tolist()}")


In [ ]:
# ─── Basic info ───────────────────────────────────────────────────────────────
print("─── Data Types & Non-Null Counts ───")
df.info()


In [ ]:
# ─── Summary statistics ───────────────────────────────────────────────────────
print("─── Summary Statistics ───")
df.describe().T


In [ ]:
# ─── Missing values ───────────────────────────────────────────────────────────
missing = df.isnull().sum()
print("─── Missing Values ───")
print(missing[missing > 0] if missing.any() else "✅ No missing values found.")


## 2. 📊 Exploratory Data Analysis (EDA)

### 2.1 Target Class Distribution

The target variable `is_fraud` is **heavily imbalanced**:
- `0` = Legitimate transaction (~99.5%)
- `1` = Fraudulent transaction (~0.5%)

This imbalance is typical in fraud detection. If we trained a naive model that always predicted "not fraud", it would achieve ~99.5% accuracy — but be completely useless. We must use appropriate metrics (**Recall, F1, ROC-AUC**) and address the imbalance via **SMOTE** or **class weights**.

**Business requirement:** The model should err on the side of caution → prioritise **high Recall** (catch as much fraud as possible, even at the cost of some false positives).


In [ ]:
# ─── Target class distribution ───────────────────────────────────────────────
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100

print("Class Counts:")
print(f"  Legitimate (0): {fraud_counts[0]:>7,}  ({fraud_pct[0]:.2f}%)")
print(f"  Fraud      (1): {fraud_counts[1]:>7,}  ({fraud_pct[1]:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5)
axes[0].set_title('Transaction Class Distribution (Count)', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontsize=10, fontweight='bold')

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.2f%%', colors=['#2ecc71', '#e74c3c'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Balance (%)', fontsize=13)

plt.suptitle('⚠️ Highly Imbalanced Dataset — Fraud = 0.52% of transactions', fontsize=12, color='#e74c3c')
plt.tight_layout()
plt.show()


### 2.2 Transaction Amount Analysis

Transaction amount (`amt`) is one of the most important fraud signals. Fraudsters often make unusually large or unusually small transactions. We compare the distribution between legitimate and fraudulent transactions.


In [ ]:
# ─── Transaction amount distribution by class ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE plot
for label, color, name in [(0, '#2ecc71', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    subset = df[df['is_fraud'] == label]['amt']
    axes[0].hist(subset.clip(upper=1000), bins=60, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('Transaction Amount Distribution (clipped at $1000)', fontsize=12)
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Box plot
df_plot = df[['amt', 'is_fraud']].copy()
df_plot['Class'] = df_plot['is_fraud'].map({0: 'Legitimate', 1: 'Fraud'})
axes[1].boxplot(
    [df[df['is_fraud']==0]['amt'].clip(upper=1500), df[df['is_fraud']==1]['amt'].clip(upper=1500)],
    labels=['Legitimate', 'Fraud'], patch_artist=True,
    boxprops=dict(facecolor='#aed6f1'), medianprops=dict(color='navy', linewidth=2)
)
axes[1].set_title('Amount Box Plot by Class (clipped at $1500)', fontsize=12)
axes[1].set_ylabel('Amount ($)')

plt.suptitle('Transaction Amount vs Fraud', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nAmount stats — Legitimate: mean=${df[df.is_fraud==0].amt.mean():.2f}, median=${df[df.is_fraud==0].amt.median():.2f}")
print(f"Amount stats — Fraud:      mean=${df[df.is_fraud==1].amt.mean():.2f}, median=${df[df.is_fraud==1].amt.median():.2f}")


### 2.3 Fraud by Merchant Category

Different spending categories have different fraud rates. Understanding which categories are most fraud-prone helps the model and gives business insight.


In [ ]:
# ─── Fraud rate by category ───────────────────────────────────────────────────
cat_fraud = df.groupby('category')['is_fraud'].agg(['sum', 'count'])
cat_fraud['fraud_rate'] = cat_fraud['sum'] / cat_fraud['count'] * 100
cat_fraud = cat_fraud.sort_values('fraud_rate', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Fraud rate %
bars = axes[0].barh(cat_fraud.index, cat_fraud['fraud_rate'], color='#e74c3c', alpha=0.8)
axes[0].set_title('Fraud Rate (%) by Category', fontsize=12)
axes[0].set_xlabel('Fraud Rate (%)')
for bar, val in zip(bars, cat_fraud['fraud_rate']):
    axes[0].text(val + 0.05, bar.get_y() + bar.get_height()/2, f'{val:.2f}%', va='center', fontsize=8)

# Transaction counts
cat_counts = df.groupby(['category', 'is_fraud']).size().unstack(fill_value=0)
cat_counts.columns = ['Legitimate', 'Fraud']
cat_counts = cat_counts.reindex(cat_fraud.index)
cat_counts.plot(kind='barh', ax=axes[1], color=['#2ecc71', '#e74c3c'], alpha=0.8, width=0.7)
axes[1].set_title('Transaction Count by Category & Class', fontsize=12)
axes[1].set_xlabel('Number of Transactions')
axes[1].legend()

plt.suptitle('Merchant Category Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 2.4 Fraud by State and Geographic Distribution

Geographic location may be a strong fraud signal — some states may have higher fraud rates due to population density or specific fraudster activity patterns.


In [ ]:
# ─── Fraud rate by state ──────────────────────────────────────────────────────
state_fraud = df.groupby('state')['is_fraud'].agg(['sum', 'count'])
state_fraud['fraud_rate'] = state_fraud['sum'] / state_fraud['count'] * 100
state_fraud = state_fraud.sort_values('fraud_rate', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top 15 states by fraud rate
axes[0].bar(state_fraud.index, state_fraud['fraud_rate'], color='#e67e22', alpha=0.85, edgecolor='white')
axes[0].set_title('Top 15 States by Fraud Rate (%)', fontsize=12)
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_xlabel('State')
axes[0].tick_params(axis='x', rotation=45)

# Scatter: lat vs long coloured by fraud
sample = df.sample(n=5000, random_state=42)
scatter = axes[1].scatter(sample['long'], sample['lat'],
                           c=sample['is_fraud'], cmap='RdYlGn_r',
                           alpha=0.4, s=5)
axes[1].set_title('Geographic Distribution (sample 5k)', fontsize=12)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
plt.colorbar(scatter, ax=axes[1], label='is_fraud (0=legit, 1=fraud)')

plt.suptitle('Geographic Fraud Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 2.5 Correlation Heatmap

A correlation heatmap of numerical features helps us identify which features are related to each other and to the target variable. High correlation with `is_fraud` indicates predictive power.


In [ ]:
# ─── Correlation heatmap ──────────────────────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 10})
plt.title('Correlation Heatmap — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Highlight correlations with target
print("\n── Correlation with is_fraud ──")
print(corr['is_fraud'].drop('is_fraud').sort_values(key=abs, ascending=False).to_string())


### 2.6 Violin & Box Plots — Numerical Feature Spread by Class

Violin and box plots help visualise the distribution and spread of key numerical features, split by class. We look for features where the fraud and legitimate distributions clearly differ.


In [ ]:
# ─── Violin plots for key numerical features ─────────────────────────────────
# Use a small stratified sample for faster plotting
sample_viz = df.groupby('is_fraud', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 1000), random_state=42)
)
sample_viz['Class'] = sample_viz['is_fraud'].map({0: 'Legitimate', 1: 'Fraud'})

features_to_plot = ['amt', 'city_pop', 'lat', 'long']
fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for i, feat in enumerate(features_to_plot):
    # Violin
    sns.violinplot(data=sample_viz, x='Class', y=feat, palette=['#2ecc71', '#e74c3c'],
                   inner='box', ax=axes[0, i])
    axes[0, i].set_title(f'Violin: {feat}', fontsize=11)
    axes[0, i].set_xlabel('')

    # Box
    sns.boxplot(data=sample_viz, x='Class', y=feat, palette=['#2ecc71', '#e74c3c'],
                ax=axes[1, i], width=0.5)
    axes[1, i].set_title(f'Box: {feat}', fontsize=11)
    axes[1, i].set_xlabel('')

plt.suptitle('Numerical Feature Distributions by Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 3. 🔧 Data Preprocessing & Feature Engineering

### Strategy

| Column | Type | Action |
|--------|------|--------|
| `trans_date_trans_time` | datetime string | Extract `hour`, `day_of_week`, `month` |
| `dob` | date string | Compute `age` at transaction time |
| `merchant` | high-cardinality categorical | Drop (too many unique values) |
| `city` | high-cardinality categorical | Drop (captured by `lat`/`long`) |
| `trans_num` | ID string | Drop |
| `job` | high-cardinality categorical | Drop |
| `category` | categorical (14 values) | One-Hot Encode |
| `state` | categorical (50 values) | One-Hot Encode |
| `amt`, `lat`, `long`, `city_pop`, `merch_lat`, `merch_long` | numerical | Keep, scale for Logistic Regression |
| `hour`, `day_of_week`, `month`, `age` | engineered numerical | Keep |

**Feature Engineering rationale:**
- `hour` of transaction: fraud often happens at night
- `age`: older/younger customers may be more vulnerable
- `distance`: distance between customer and merchant location (Haversine approximation) — unusual if very far
- `log_amt`: log-transform of amount to reduce skewness


In [ ]:
# ─── Feature Engineering ──────────────────────────────────────────────────────
df_fe = df.copy()

# 1. Parse datetime → extract time features
df_fe['trans_date_trans_time'] = pd.to_datetime(df_fe['trans_date_trans_time'])
df_fe['hour']        = df_fe['trans_date_trans_time'].dt.hour
df_fe['day_of_week'] = df_fe['trans_date_trans_time'].dt.dayofweek   # 0=Mon, 6=Sun
df_fe['month']       = df_fe['trans_date_trans_time'].dt.month

# 2. Compute customer age at time of transaction
df_fe['dob'] = pd.to_datetime(df_fe['dob'])
df_fe['age'] = (df_fe['trans_date_trans_time'] - df_fe['dob']).dt.days // 365

# 3. Distance between customer and merchant (approximate Haversine)
def haversine_approx(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

df_fe['distance_km'] = haversine_approx(
    df_fe['lat'], df_fe['long'],
    df_fe['merch_lat'], df_fe['merch_long']
)

# 4. Log-transform of amount (reduces right-skew)
df_fe['log_amt'] = np.log1p(df_fe['amt'])

# 5. Drop columns not useful for modelling
drop_cols = ['trans_date_trans_time', 'dob', 'merchant', 'city', 'trans_num', 'job']
df_fe.drop(columns=drop_cols, inplace=True)

print("Shape after feature engineering:", df_fe.shape)
print("\nColumns:", df_fe.columns.tolist())
df_fe.head(3)


In [ ]:
# ─── One-Hot Encode categorical features ─────────────────────────────────────
# 'category' has 14 unique values, 'state' has ~30 — both manageable for OHE
df_encoded = pd.get_dummies(df_fe, columns=['category', 'state'], drop_first=False, dtype=int)

print("Shape after One-Hot Encoding:", df_encoded.shape)
print("\nNew columns added:", [c for c in df_encoded.columns if c.startswith('category_') or c.startswith('state_')])


## 4. ✂️ Train–Test Split & Class Imbalance Handling

### Split Strategy
- **80% training / 20% test** with `stratify=y` to maintain class ratio in both sets
- `random_state=42` for full reproducibility
- ⚠️ The test set is **locked** — never used for model selection or tuning

### Class Imbalance Strategy
The dataset has ~0.52% fraud. We apply **SMOTE (Synthetic Minority Over-sampling Technique)** only on the **training set** to balance classes before model training.  
> SMOTE creates synthetic samples of the minority class rather than simply duplicating — it interpolates between existing minority samples, producing more diverse and realistic synthetic fraud transactions.

**Why SMOTE over class_weight?** SMOTE gives the model more variety to learn from. We'll also try `class_weight='balanced'` for Logistic Regression as a comparison.


In [ ]:
# ─── Train–Test Split ────────────────────────────────────────────────────────
X = df_encoded.drop(columns=['is_fraud'])
y = df_encoded['is_fraud']

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set : {X_train_raw.shape[0]:>7,} rows  ({y_train_raw.mean()*100:.2f}% fraud)")
print(f"Test set     : {X_test.shape[0]:>7,} rows  ({y_test.mean()*100:.2f}% fraud)")

# ─── Scale numerical features (needed for Logistic Regression) ───────────────
num_features = ['amt', 'log_amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long',
                'hour', 'day_of_week', 'month', 'age', 'distance_km']
num_features = [f for f in num_features if f in X_train_raw.columns]  # safety check

scaler = StandardScaler()
X_train_scaled = X_train_raw.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[num_features] = scaler.fit_transform(X_train_raw[num_features])
X_test_scaled[num_features]  = scaler.transform(X_test[num_features])

print("\nScaling applied to:", num_features)

# ─── SMOTE on training set only ───────────────────────────────────────────────
print("\n⏳ Applying SMOTE to training set (this may take ~1 min)...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_raw, y_train_raw)

print(f"\nAfter SMOTE:")
print(f"  Legitimate: {(y_train_sm==0).sum():>7,}")
print(f"  Fraud     : {(y_train_sm==1).sum():>7,}")
print(f"  Total     : {len(y_train_sm):>7,}")


## 5. 🤖 Model Training & Hyperparameter Tuning

We train **5 models** using `StratifiedKFold` cross-validation on the SMOTE-balanced training set:

| # | Model | Notes |
|---|-------|-------|
| 1 | **Logistic Regression** | Linear baseline; needs scaled features; `class_weight='balanced'` |
| 2 | **Decision Tree** | Interpretable; no scaling needed |
| 3 | **Random Forest** | Ensemble of trees; robust to noise |
| 4 | **XGBoost** | Gradient boosting; typically best on tabular data |
| 5 | **LightGBM** ⭐ Bonus | Fast gradient boosting; handles imbalance natively |

**Tuning:** `RandomizedSearchCV` (cv=5, scoring=`f1`) to maximise fraud detection.  
**Key metric:** F1-score on the fraud class — balances Precision and Recall.  
Since the business requires high Recall, we also monitor recall separately.


In [ ]:
# ─── Model 1: Logistic Regression ────────────────────────────────────────────
# Uses scaled+SMOTE data; class_weight='balanced' as additional safeguard
print("Training Logistic Regression...")

# Scale the SMOTE data
X_train_sm_scaled = X_train_sm.copy()
X_train_sm_scaled[num_features] = scaler.transform(X_train_sm[num_features])

lr_params = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [500]
}
lr_cv = GridSearchCV(
    LogisticRegression(random_state=42, class_weight='balanced'),
    lr_params, cv=StratifiedKFold(5), scoring='f1', n_jobs=-1, verbose=0
)
lr_cv.fit(X_train_sm_scaled, y_train_sm)
best_lr = lr_cv.best_estimator_

print(f"✅ Best params: {lr_cv.best_params_}")
print(f"   Best CV F1 : {lr_cv.best_score_:.4f}")


In [ ]:
# ─── Model 2: Decision Tree ──────────────────────────────────────────────────
print("Training Decision Tree...")

dt_params = {
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 10, 50],
    'criterion': ['gini', 'entropy']
}
dt_cv = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    dt_params, n_iter=12, cv=StratifiedKFold(5), scoring='f1',
    random_state=42, n_jobs=-1, verbose=0
)
dt_cv.fit(X_train_sm, y_train_sm)
best_dt = dt_cv.best_estimator_

print(f"✅ Best params: {dt_cv.best_params_}")
print(f"   Best CV F1 : {dt_cv.best_score_:.4f}")


In [ ]:
# ─── Model 3: Random Forest ──────────────────────────────────────────────────
print("Training Random Forest (this may take 2–5 min)...")

rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 10]
}
rf_cv = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    rf_params, n_iter=10, cv=StratifiedKFold(5), scoring='f1',
    random_state=42, n_jobs=-1, verbose=1
)
rf_cv.fit(X_train_sm, y_train_sm)
best_rf = rf_cv.best_estimator_

print(f"\n✅ Best params: {rf_cv.best_params_}")
print(f"   Best CV F1 : {rf_cv.best_score_:.4f}")


In [ ]:
# ─── Model 4: XGBoost ────────────────────────────────────────────────────────
print("Training XGBoost...")

# Compute scale_pos_weight for native imbalance handling (alternative to SMOTE)
scale_pos = (y_train_raw == 0).sum() / (y_train_raw == 1).sum()

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [4, 6, 8],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 1.0]
}
xgb_cv = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss',
                  scale_pos_weight=scale_pos, tree_method='hist'),
    xgb_params, n_iter=15, cv=StratifiedKFold(5), scoring='f1',
    random_state=42, n_jobs=-1, verbose=1
)
xgb_cv.fit(X_train_raw, y_train_raw)  # XGBoost uses scale_pos_weight, not SMOTE
best_xgb = xgb_cv.best_estimator_

print(f"\n✅ Best params: {xgb_cv.best_params_}")
print(f"   Best CV F1 : {xgb_cv.best_score_:.4f}")


In [ ]:
# ─── Model 5 (Bonus): LightGBM ───────────────────────────────────────────────
print("Training LightGBM (Bonus model)...")

lgbm_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [-1, 6, 10],
    'num_leaves': [31, 63, 127],
    'min_child_samples': [20, 50]
}
lgbm_cv = RandomizedSearchCV(
    LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1),
    lgbm_params, n_iter=15, cv=StratifiedKFold(5), scoring='f1',
    random_state=42, n_jobs=-1, verbose=0
)
lgbm_cv.fit(X_train_raw, y_train_raw)
best_lgbm = lgbm_cv.best_estimator_

print(f"✅ Best params: {lgbm_cv.best_params_}")
print(f"   Best CV F1 : {lgbm_cv.best_score_:.4f}")


In [ ]:
# ─── Best hyperparameters summary table ──────────────────────────────────────
param_summary = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Best Parameters': str(lr_cv.best_params_),  'CV F1': round(lr_cv.best_score_, 4)},
    {'Model': 'Decision Tree',       'Best Parameters': str(dt_cv.best_params_),  'CV F1': round(dt_cv.best_score_, 4)},
    {'Model': 'Random Forest',       'Best Parameters': str(rf_cv.best_params_),  'CV F1': round(rf_cv.best_score_, 4)},
    {'Model': 'XGBoost',             'Best Parameters': str(xgb_cv.best_params_), 'CV F1': round(xgb_cv.best_score_, 4)},
    {'Model': 'LightGBM (Bonus)',    'Best Parameters': str(lgbm_cv.best_params_),'CV F1': round(lgbm_cv.best_score_, 4)},
])
param_summary = param_summary.sort_values('CV F1', ascending=False).reset_index(drop=True)
print("─── Best Hyperparameters per Model ───")
param_summary


## 6. 📈 Evaluation & Comparison

All tuned models are now evaluated on the **held-out test set** (never seen during training or tuning).

**Metrics reported:**
- **Accuracy** — overall correct predictions (misleading due to imbalance, reported for completeness)
- **Precision** — of all predicted fraud, how many are truly fraud
- **Recall** — of all actual fraud, how many did we catch ← **most important** given business requirement
- **F1-Score** — harmonic mean of Precision and Recall
- **ROC-AUC** — overall discriminative power of the model

> ⚠️ Remember: a model that predicts everything as "not fraud" achieves 99.5% accuracy but 0% recall. Accuracy alone is **not** a valid metric here.


In [ ]:
# ─── Evaluate all models on test set ─────────────────────────────────────────
models = {
    'Logistic Regression': (best_lr, X_test_scaled),
    'Decision Tree':       (best_dt, X_test),
    'Random Forest':       (best_rf, X_test),
    'XGBoost':             (best_xgb, X_test),
    'LightGBM':            (best_lgbm, X_test),
}

results = []
roc_data = {}

for name, (model, X_eval) in models.items():
    y_pred  = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    roc  = roc_auc_score(y_test, y_proba)

    results.append({'Model': name, 'Accuracy': round(acc, 4),
                    'Precision': round(prec, 4), 'Recall': round(rec, 4),
                    'F1-Score': round(f1, 4), 'ROC-AUC': round(roc, 4)})

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, roc)

    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)


In [ ]:
# ─── Summary comparison table ────────────────────────────────────────────────
print("═══ Model Comparison Summary ═══")
results_df.style.background_gradient(
    subset=['Recall', 'F1-Score', 'ROC-AUC'], cmap='YlGn'
).background_gradient(
    subset=['Accuracy'], cmap='Blues'
).format({
    'Accuracy': '{:.4f}', 'Precision': '{:.4f}',
    'Recall': '{:.4f}', 'F1-Score': '{:.4f}', 'ROC-AUC': '{:.4f}'
})


In [ ]:
# ─── Grouped bar chart: Accuracy / F1 / ROC-AUC ─────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(results_df))
width = 0.15
colors = ['#3498db', '#9b59b6', '#e74c3c', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.85, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xticks(x + 2*width)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.axhline(0.5, color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# ─── ROC Curves — all models on one plot ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6']

for (name, (fpr, tpr, roc_val)), color in zip(roc_data.items(), colors_roc):
    ax.plot(fpr, tpr, label=f'{name}  (AUC = {roc_val:.4f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6, label='Random (AUC = 0.50)')
ax.set_xlim([-0.01, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ─── Confusion Matrix heatmaps — best & worst model ──────────────────────────
best_model_name  = results_df.iloc[0]['Model']
worst_model_name = results_df.iloc[-1]['Model']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model_name, title_color in [
    (axes[0], best_model_name,  '#27ae60'),
    (axes[1], worst_model_name, '#c0392b')
]:
    model, X_eval = models[model_name]
    y_pred = model.predict(X_eval)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['Predicted: Legit', 'Predicted: Fraud'],
                yticklabels=['Actual: Legit', 'Actual: Fraud'],
                linewidths=0.5)
    ax.set_title(f'{"🏆 BEST" if model_name == best_model_name else "📉 WORST"}: {model_name}',
                 fontsize=12, color=title_color, fontweight='bold')
    tn, fp, fn, tp = cm.ravel()
    ax.set_xlabel(f'TP={tp:,} | FP={fp:,} | FN={fn:,} | TN={tn:,}', fontsize=9)

plt.suptitle('Confusion Matrices — Best vs Worst Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. 🔍 Model Interpretation & Feature Importance

### Why does the best model win?

After comparing all models, we perform an in-depth analysis of the winning model:
- **Feature importances** — which features drive fraud predictions
- **Discussion** of model properties (complexity, ensemble effects, regularisation)
- **Domain interpretation** — do the important features make real-world sense?

Feature importance from tree-based models gives us a direct measure of how much each feature reduces impurity (Gini or entropy) across all trees.


In [ ]:
# ─── Feature importances — best tree-based model ─────────────────────────────
# Identify best tree-based model (XGBoost or Random Forest or LightGBM)
tree_models_ordered = results_df[results_df['Model'].isin(['XGBoost', 'Random Forest', 'LightGBM'])]
best_tree_name = tree_models_ordered.iloc[0]['Model'] if len(tree_models_ordered) > 0 else best_model_name
best_tree_model, best_tree_X = models[best_tree_name]

print(f"Feature importance from: {best_tree_name}")

importances = best_tree_model.feature_importances_
feature_names = X_test.columns.tolist()
feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values('Importance', ascending=True).tail(25)

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
               color=plt.cm.RdYlGn(feat_imp_df['Importance'] / feat_imp_df['Importance'].max()),
               edgecolor='white')
ax.set_title(f'Top 25 Feature Importances — {best_tree_name}', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, feat_imp_df['Importance']):
    ax.text(val + 0.0003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# ─── Feature importances — Random Forest (for comparison) ────────────────────
rf_imp_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(rf_imp_df['Feature'], rf_imp_df['Importance'],
        color='#3498db', alpha=0.8, edgecolor='white')
ax.set_title('Top 20 Feature Importances — Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()


### 7.1 Domain Interpretation

**Expected important features and why they make sense:**

| Feature | Why it matters |
|---------|---------------|
| `amt` / `log_amt` | Fraudsters often make unusually large purchases; amount is one of the strongest signals |
| `distance_km` | If a customer's card is used far from their home location, it's suspicious |
| `hour` | Fraud often occurs at night or early morning when monitoring is reduced |
| `category_*` | Certain categories (shopping, grocery_net) have higher fraud rates |
| `city_pop` | Population density correlates with fraud exposure |
| `age` | Older customers may be targeted more by certain fraud types |
| `lat` / `long` / `merch_lat` / `merch_long` | Geographic anomalies signal fraudulent activity |

**Model complexity analysis:**
- **Logistic Regression**: Linear decision boundary — can capture basic linear relationships but misses non-linear interactions between `amt`, `hour`, and `category`. Lowest performer.
- **Decision Tree**: Single tree — captures non-linear splits but prone to overfitting. Moderate performance.
- **Random Forest**: 100+ trees, each on a random subset → low variance through averaging. Robust and strong.
- **XGBoost / LightGBM**: Gradient boosting — each new tree focuses on the errors of the previous ones. Best at capturing subtle fraud patterns. Typically the winner on tabular data.


## 8. 🏁 Conclusions

### Summary of Findings

| Question | Answer |
|----------|--------|
| Best model overall? | See results table above — XGBoost/LightGBM expected to lead |
| Most important fraud signal? | Transaction amount, distance, and hour of day |
| How did we handle imbalance? | SMOTE on training set + class_weight='balanced' |
| Does the model meet the business requirement? | Focus on Recall — better to flag legitimate transactions than miss fraud |

### Key Takeaways

1. **Imbalance is the central challenge** — 99.5% legitimate transactions. Standard accuracy is meaningless here; Recall and ROC-AUC are the real metrics.
2. **Feature engineering matters** — Derived features like `distance_km`, `hour`, `age`, and `log_amt` provided the model with richer signals that raw columns alone could not.
3. **Ensemble methods dominate tabular data** — XGBoost and LightGBM outperformed simpler models (Logistic Regression, Decision Tree) because they can capture non-linear interactions.
4. **Logistic Regression is a useful baseline** — its relative simplicity makes it easy to interpret, and its performance gap with ensembles quantifies the value of complexity.

### What We Would Explore with More Time

- **SHAP values** for individual prediction explanations and better feature interaction understanding
- **Threshold optimisation** — since we prioritise Recall, we could lower the classification threshold from 0.5 to e.g. 0.3 to catch more fraud at the cost of more false positives
- **Time-series cross-validation** — transactions have temporal order; using time-based splits would be more realistic than random splits
- **Deep learning (TabNet)** — to compare against tree models on this dataset

### References

- Dataset: [Credit Card Fraud Dataset on Kaggle](https://www.kaggle.com/datasets/dhruvb2028/credit-card-fraud-dataset)
- Shwartz-Ziv & Armon (2021): *"Tabular Data: Deep Learning is Not All You Need"* — https://arxiv.org/abs/2106.03253
- Scikit-learn Documentation: https://scikit-learn.org/stable/
- XGBoost Documentation: https://xgboost.readthedocs.io/
- LightGBM Documentation: https://lightgbm.readthedocs.io/
- SMOTE: Chawla et al. (2002) — imbalanced-learn library

---
*Notebook prepared for Basics of AI and Machine Learning — Final Project, April 2026.*
